# Chapter 9: Reward Models — The GAS Safety Critic

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# !pip install sentence-transformers numpy matplotlib scikit-learn

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Understand **reward model architecture** and training objectives
- Build a **MedicalSafetyRewardModel** using sentence transformers as backbone
- Train on **20 (prompt, chosen, rejected) pairs** of medical advice
- Evaluate on **5 test examples** and interpret scores
- Understand **reward hacking** and how to detect it

---
## 📖 2. Theory Recap

### What is a Reward Model?

A reward model $R_\phi(x, y)$ maps a (prompt, response) pair to a scalar score:

$$R_\phi: (x, y) \rightarrow \mathbb{R}$$

It's trained on human preference data using the **Bradley-Terry model**:

$$P(y_w \succ y_l | x) = \sigma(R_\phi(x, y_w) - R_\phi(x, y_l))$$

Training objective (maximize log-likelihood of preferences):

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma(R_\phi(x, y_w) - R_\phi(x, y_l)) \right]$$

### Reward Model Architecture

```
Input: (prompt, response)
    ↓
Encoder (e.g., sentence-transformers)
    ↓
Embedding vector [768-dim]
    ↓
Linear head → scalar score
```

### Reward Hacking

A key challenge: the policy can find responses that **score high but are actually bad**. This is called reward hacking or Goodhart's Law:

> "When a measure becomes a target, it ceases to be a good measure."

Mitigations:
- KL penalty against reference model
- Ensemble of reward models
- Regular human evaluation
- Constitutional AI constraints

---
## 🔨 3. Build It

### 3.1 Medical Safety Reward Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple

# Safe medical advice examples (positive class)
SAFE_EXAMPLES = [
    "Check for ketones, contact doctor if glucose above 300, drink water, monitor closely",
    "Treat hypoglycemia with 15g fast-acting carbs, recheck in 15 minutes, repeat if needed",
    "Never stop insulin without physician guidance, schedule medication review immediately",
    "Sick day rules: continue insulin even when not eating, monitor every 2-4 hours",
    "Non-healing diabetic foot wound requires immediate podiatrist referral, infection risk",
    "HbA1c 9.8% indicates poor glycemic control, urgent medication review and lifestyle intervention needed",
    "Alcohol can cause hypoglycemia with insulin, eat food, monitor glucose, limit intake",
    "Nocturnal hypoglycemia: reduce evening basal insulin, bedtime snack with protein, set CGM alerts",
    "Glucose 400 with nausea may indicate DKA emergency, check ketones, go to ER if positive",
    "Never share insulin pens due to infection risk including HIV and hepatitis",
]

# Unsafe medical advice examples (negative class)
UNSAFE_EXAMPLES = [
    "Take extra insulin immediately without checking ketones or consulting a doctor",
    "If you feel fine just skip your medication, it's probably not necessary",
    "Double your dose to compensate for the missed one, it will balance out",
    "Alcohol is fine with diabetes medication, no interaction to worry about",
    "The foot wound will heal on its own, just keep it bandaged",
    "HbA1c of 9.8 is a bit high but nothing serious, just eat less sugar",
    "If you're not eating, skip your insulin to avoid hypoglycemia",
    "Blood sugar 400 with nausea is normal, take insulin and rest",
    "You can borrow a friend's insulin pen if you use a new needle",
    "Stop insulin if glucose is 140 and you feel fine, you might not need it",
]

print(f"Safe examples: {len(SAFE_EXAMPLES)}")
print(f"Unsafe examples: {len(UNSAFE_EXAMPLES)}")
print("\nExample safe advice:")
print(f"  '{SAFE_EXAMPLES[0]}'")
print("\nExample unsafe advice:")
print(f"  '{UNSAFE_EXAMPLES[0]}'")

In [ ]:
class MedicalSafetyRewardModel:
    """
    A reward model for medical advice safety scoring.
    
    Architecture:
    - Backbone: SentenceTransformer for semantic embeddings
    - Scoring: cosine similarity to safe/unsafe centroids
    - Output: safety score in [0, 1]
    """
    
    def __init__(self, test_mode: bool = False):
        self.test_mode = test_mode
        self.safe_centroid = None
        self.unsafe_centroid = None
        self.is_trained = False
        
        if not test_mode:
            try:
                from sentence_transformers import SentenceTransformer
                self.model = SentenceTransformer('all-MiniLM-L6-v2')
                print("✅ Loaded SentenceTransformer: all-MiniLM-L6-v2")
            except ImportError:
                print("⚠️  sentence-transformers not installed, using mock embeddings")
                self.model = None
                self.test_mode = True
        else:
            self.model = None
            print("TEST_MODE: Using mock embeddings")
    
    def _embed(self, texts: List[str]) -> np.ndarray:
        """Embed texts using the backbone model."""
        if self.model is not None:
            return self.model.encode(texts, normalize_embeddings=True)
        else:
            # Mock embeddings for testing
            np.random.seed(42)
            return np.random.randn(len(texts), 384)
    
    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> float:
        """Compute cosine similarity between two vectors."""
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))
    
    def fit(self, safe_examples: List[str], unsafe_examples: List[str]):
        """
        Train by computing centroids of safe and unsafe embeddings.
        """
        print("Training MedicalSafetyRewardModel...")
        
        # Embed all examples
        safe_embeddings = self._embed(safe_examples)
        unsafe_embeddings = self._embed(unsafe_examples)
        
        # Compute centroids
        self.safe_centroid = np.mean(safe_embeddings, axis=0)
        self.unsafe_centroid = np.mean(unsafe_embeddings, axis=0)
        
        # Normalize centroids
        self.safe_centroid /= np.linalg.norm(self.safe_centroid)
        self.unsafe_centroid /= np.linalg.norm(self.unsafe_centroid)
        
        self.is_trained = True
        print(f"  Trained on {len(safe_examples)} safe + {len(unsafe_examples)} unsafe examples")
        print(f"  Safe centroid norm: {np.linalg.norm(self.safe_centroid):.4f}")
        print(f"  Unsafe centroid norm: {np.linalg.norm(self.unsafe_centroid):.4f}")
        
        # Compute centroid separation
        separation = self._cosine_similarity(self.safe_centroid, self.unsafe_centroid)
        print(f"  Safe-unsafe centroid similarity: {separation:.4f} (lower = better separation)")
        return self
    
    def score(self, prompt: str, response: str) -> float:
        """
        Score a (prompt, response) pair for medical safety.
        
        Returns: score in [0, 1] where 1 = very safe, 0 = very unsafe
        """
        if not self.is_trained:
            raise RuntimeError("Model not trained. Call fit() first.")
        
        # Embed the combined prompt+response
        combined = f"Prompt: {prompt} Response: {response}"
        embedding = self._embed([combined])[0]
        embedding = embedding / (np.linalg.norm(embedding) + 1e-8)
        
        # Compute similarities to centroids
        sim_safe = self._cosine_similarity(embedding, self.safe_centroid)
        sim_unsafe = self._cosine_similarity(embedding, self.unsafe_centroid)
        
        # Convert to probability using softmax-like normalization
        score = np.exp(sim_safe) / (np.exp(sim_safe) + np.exp(sim_unsafe))
        return float(score)
    
    def score_batch(self, pairs: List[Tuple[str, str]]) -> List[float]:
        """Score a batch of (prompt, response) pairs."""
        return [self.score(p, r) for p, r in pairs]


# Initialize and train the model
reward_model = MedicalSafetyRewardModel(test_mode=TEST_MODE)
reward_model.fit(SAFE_EXAMPLES, UNSAFE_EXAMPLES)

### 3.2 Training Dataset (20 Preference Pairs)

In [ ]:
# 20 training preference pairs
training_pairs = [
    {"prompt": "Patient glucose 280 mg/dL, what to do?",
     "chosen": "Check for ketones, contact doctor if >300, drink water, light walk if feeling well",
     "rejected": "Take extra insulin immediately without checking"},
    {"prompt": "Glucose 55 mg/dL, patient feels shaky",
     "chosen": "Treat hypoglycemia: 15g fast-acting carbs, recheck in 15 min, repeat if still <70",
     "rejected": "Patient should lie down and rest until they feel better"},
    {"prompt": "Patient missed metformin dose this morning",
     "chosen": "Skip missed dose, take next scheduled dose. Never double up metformin.",
     "rejected": "Take double the dose now to compensate"},
    {"prompt": "HbA1c came back at 9.8%, is this serious?",
     "chosen": "Yes, 9.8% indicates poor glycemic control. Target is <7%. Urgent medication review needed.",
     "rejected": "It's a bit high but nothing to worry about, just eat less sugar"},
    {"prompt": "Patient on insulin: glucose 400 with nausea and vomiting",
     "chosen": "May indicate DKA — medical emergency. Check ketones immediately. If positive, go to ER.",
     "rejected": "Take extra insulin and drink plenty of water, the nausea will pass"},
    {"prompt": "Can patient drink alcohol with diabetes medication?",
     "chosen": "Alcohol can cause hypoglycemia with insulin. Eat food, monitor glucose, limit to 1-2 drinks.",
     "rejected": "Alcohol is fine in moderation, diabetes medication doesn't interact with it"},
    {"prompt": "Patient wants to stop insulin because they feel fine",
     "chosen": "Never stop insulin without physician guidance. Schedule a medication review.",
     "rejected": "If you feel fine, you can try stopping insulin for a few days"},
    {"prompt": "Diabetic foot wound not healing after 2 weeks",
     "chosen": "Non-healing diabetic foot wounds are serious. Refer to podiatrist immediately. Risk of amputation.",
     "rejected": "Keep it clean and bandaged, it will heal on its own eventually"},
    {"prompt": "Patient sick with flu and not eating. Skip insulin?",
     "chosen": "Never skip insulin when sick. Illness raises glucose. Monitor every 2-4 hours, check ketones.",
     "rejected": "If you're not eating, skip your insulin dose to avoid hypoglycemia"},
    {"prompt": "Patient asks about using friend's insulin pen",
     "chosen": "Never share insulin pens — serious infection risk including HIV and hepatitis.",
     "rejected": "It should be fine to borrow a pen if it's a new needle"},
    {"prompt": "Frequent nighttime hypoglycemia at 3am",
     "chosen": "Reduce evening basal insulin, bedtime snack with protein, set CGM alerts, discuss with endocrinologist",
     "rejected": "Eat more before bed to prevent it, no need to change medication"},
    {"prompt": "Glucose 350 before surgery tomorrow",
     "chosen": "Notify surgical team immediately. Surgery with glucose >250 increases infection risk. May need postponement.",
     "rejected": "Take your regular insulin and proceed with surgery as planned"},
    {"prompt": "Type 1 diabetic wants to run a marathon",
     "chosen": "Consult endocrinologist. Reduce basal insulin 20-30%, carry fast carbs, monitor every 30 min.",
     "rejected": "Just run normally, your body will adapt to the exercise"},
    {"prompt": "Patient with Type 2 wants to start intermittent fasting",
     "chosen": "Consult endocrinologist first. IF requires medication adjustment to prevent hypoglycemia.",
     "rejected": "Intermittent fasting is great for everyone, just skip breakfast"},
    {"prompt": "CGM showing rapid drop from 180 to 90 in 30 minutes",
     "chosen": "Eat 15g carbs preventively, recheck in 15 min. Investigate cause: exercise, insulin timing.",
     "rejected": "90 mg/dL is normal, no action needed"},
    {"prompt": "Patient has blurry vision and glucose is 350",
     "chosen": "Blurry vision with high glucose may indicate diabetic retinopathy risk. Ophthalmology referral needed.",
     "rejected": "Blurry vision is normal when tired, just rest your eyes"},
    {"prompt": "Patient wants to use herbal supplements instead of metformin",
     "chosen": "Herbal supplements are not proven to control diabetes. Do not replace prescribed medication without physician guidance.",
     "rejected": "Natural remedies are safer than medication, try cinnamon and berberine instead"},
    {"prompt": "Patient's glucose is consistently 200-250 despite medication",
     "chosen": "Persistent hyperglycemia despite medication requires urgent endocrinology review and possible medication adjustment.",
     "rejected": "Just exercise more and eat less, the medication will eventually work"},
    {"prompt": "Patient has numbness in feet",
     "chosen": "Foot numbness in diabetics may indicate peripheral neuropathy. Neurological evaluation and foot care education needed.",
     "rejected": "Numbness is probably from sitting too long, just walk around"},
    {"prompt": "Patient asks if they can eat fruit with diabetes",
     "chosen": "Fruit is healthy but contains carbohydrates. Monitor glucose response, prefer low-GI fruits, count carbs in meal plan.",
     "rejected": "Diabetics should never eat fruit because it contains sugar"},
]

print(f"Training dataset: {len(training_pairs)} preference pairs")

# Score chosen vs rejected for all training pairs
chosen_scores = []
rejected_scores = []

for pair in training_pairs:
    cs = reward_model.score(pair['prompt'], pair['chosen'])
    rs = reward_model.score(pair['prompt'], pair['rejected'])
    chosen_scores.append(cs)
    rejected_scores.append(rs)

print(f"\nTraining set evaluation:")
print(f"  Mean chosen score:   {np.mean(chosen_scores):.4f}")
print(f"  Mean rejected score: {np.mean(rejected_scores):.4f}")
print(f"  Correct rankings:    {sum(c > r for c, r in zip(chosen_scores, rejected_scores))}/{len(training_pairs)}")

### 3.3 Evaluation on Test Examples

In [ ]:
# 5 test examples (not seen during training)
test_examples = [
    {
        "prompt": "Patient with Type 2 diabetes has chest pain",
        "response": "Chest pain in diabetics may be a cardiac emergency. Call 911 immediately. Diabetics often have atypical MI symptoms.",
        "expected": "safe"
    },
    {
        "prompt": "Patient with Type 2 diabetes has chest pain",
        "response": "It's probably just indigestion, take an antacid and lie down.",
        "expected": "unsafe"
    },
    {
        "prompt": "Patient glucose is 65 mg/dL before exercise",
        "response": "Glucose of 65 is borderline low before exercise. Eat 15-20g carbs before starting, monitor during exercise.",
        "expected": "safe"
    },
    {
        "prompt": "Patient glucose is 65 mg/dL before exercise",
        "response": "65 is fine, just start exercising and you'll be okay.",
        "expected": "unsafe"
    },
    {
        "prompt": "Patient asks about continuous glucose monitoring",
        "response": "CGM provides real-time glucose data every 5 minutes, helps identify patterns, reduces hypoglycemia risk. Discuss with your endocrinologist.",
        "expected": "safe"
    },
]

print("Test Set Evaluation")
print("=" * 70)
print(f"{'#':>3} {'Expected':>10} {'Score':>8} {'Correct':>8}  Response Preview")
print("-" * 70)

test_scores = []
for i, ex in enumerate(test_examples):
    score = reward_model.score(ex['prompt'], ex['response'])
    test_scores.append(score)
    
    predicted = "safe" if score > 0.5 else "unsafe"
    correct = "✅" if predicted == ex['expected'] else "❌"
    preview = ex['response'][:50] + "..."
    print(f"{i+1:>3} {ex['expected']:>10} {score:>8.4f} {correct:>8}  {preview}")

correct_count = sum(
    ("safe" if s > 0.5 else "unsafe") == ex['expected'] 
    for s, ex in zip(test_scores, test_examples)
)
print(f"\nAccuracy: {correct_count}/{len(test_examples)} ({100*correct_count/len(test_examples):.0f}%)")

### 3.4 Reward Hacking Demo

In [ ]:
# Reward hacking: responses that score high but are actually unsafe
# A model might learn to include safety keywords without actually being safe

hacking_examples = [
    {
        "name": "Keyword stuffing",
        "prompt": "Patient glucose 400 with vomiting",
        "response": "Check ketones, monitor glucose, contact doctor, DKA emergency, insulin, lifestyle, HbA1c — but actually just take some insulin and rest at home.",
        "actually_safe": False,
        "why_hacking": "Contains safety keywords but gives dangerous advice at the end"
    },
    {
        "name": "Vague safety language",
        "prompt": "Should patient double insulin dose?",
        "response": "Always consult your doctor and monitor your glucose levels carefully before making any medication changes.",
        "actually_safe": True,  # This one is actually safe
        "why_hacking": "Sounds safe but gives no actionable guidance"
    },
    {
        "name": "False urgency",
        "prompt": "Patient glucose is 130 mg/dL",
        "response": "EMERGENCY: Glucose 130 requires immediate insulin injection and ER visit. This is a medical crisis!",
        "actually_safe": False,
        "why_hacking": "130 mg/dL is slightly elevated but NOT an emergency — this causes unnecessary panic"
    },
]

print("Reward Hacking Examples")
print("=" * 70)
print("These responses may score high but are actually problematic:\n")

for ex in hacking_examples:
    score = reward_model.score(ex['prompt'], ex['response'])
    print(f"Example: {ex['name']}")
    print(f"  Prompt:   {ex['prompt']}")
    print(f"  Response: {ex['response'][:80]}...")
    print(f"  Score:    {score:.4f} ({'HIGH' if score > 0.6 else 'LOW'})")
    print(f"  Actually safe: {ex['actually_safe']}")
    print(f"  Issue: {ex['why_hacking']}")
    print()

print("\n⚠️  Reward hacking mitigation strategies:")
print("  1. Ensemble multiple reward models")
print("  2. Add KL penalty against reference model")
print("  3. Regular human evaluation of high-scoring responses")
print("  4. Constitutional AI: explicit safety rules as hard constraints")
print("  5. Red-teaming: actively search for hacking examples")

### 3.5 Visualization: Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Chosen vs Rejected scores on training set
ax1 = axes[0]
x = np.arange(len(training_pairs))
ax1.scatter(x, chosen_scores, color='#2ecc71', s=60, label='Chosen (safe)', zorder=3)
ax1.scatter(x, rejected_scores, color='#e74c3c', s=60, label='Rejected (unsafe)', zorder=3)
for i in range(len(training_pairs)):
    ax1.plot([i, i], [chosen_scores[i], rejected_scores[i]], 'gray', alpha=0.4, linewidth=1)
ax1.axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label='Decision boundary')
ax1.set_title('Reward Model: Chosen vs Rejected Scores', fontsize=12, fontweight='bold')
ax1.set_xlabel('Training Example')
ax1.set_ylabel('Safety Score')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1)

# Plot 2: Score distributions
ax2 = axes[1]
ax2.hist(chosen_scores, bins=8, alpha=0.7, color='#2ecc71', label='Chosen (safe)', edgecolor='black')
ax2.hist(rejected_scores, bins=8, alpha=0.7, color='#e74c3c', label='Rejected (unsafe)', edgecolor='black')
ax2.axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Decision boundary')
ax2.set_title('Score Distribution: Safe vs Unsafe Responses', fontsize=12, fontweight='bold')
ax2.set_xlabel('Safety Score')
ax2.set_ylabel('Count')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('MedicalSafetyRewardModel Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reward_model_scores.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nModel Performance Summary:")
print(f"  Chosen scores:   mean={np.mean(chosen_scores):.3f}, std={np.std(chosen_scores):.3f}")
print(f"  Rejected scores: mean={np.mean(rejected_scores):.3f}, std={np.std(rejected_scores):.3f}")
print(f"  Score gap:       {np.mean(chosen_scores) - np.mean(rejected_scores):.3f}")

---
## 🏥 4. GAS Connection

### The GAS Safety Critic Agent

**This IS the GAS Safety Critic agent.** Every recommendation generated by the Advisor Agent passes through this reward model before being delivered to the patient.

```
GAS Safety Pipeline:

Patient Data
    ↓
[Monitor Agent] → detects anomalies
    ↓
[Advisor Agent] → generates recommendation
    ↓
[Safety Critic] ← MedicalSafetyRewardModel
    ↓
Score > 0.7? → Deliver to patient
Score < 0.7? → Escalate to human physician
```

**Safety threshold (matches gas_system.py):**
- Score ≥ 0.7: Recommendation approved → deliver to patient
- Score < 0.7: Recommendation rejected → Advisor Agent retries (up to 2×), then escalates to caregiver

**The reward model is also used during RLHF training** to provide reward signals for the Advisor Agent's policy optimization.

---
## ✅ 5. Checkpoint

Test your understanding:

**Q1: What is the Bradley-Terry model used for in reward model training?**  
A: The Bradley-Terry model provides a probabilistic framework for pairwise comparisons. It models the probability that response A is preferred over response B as σ(R(A) - R(B)), allowing us to train the reward model using binary preference labels.

**Q2: Why use a sentence transformer as the backbone instead of a full LLM?**  
A: Sentence transformers are much smaller (22M-110M params vs 7B+) and faster to run. For a safety critic that must evaluate every response in real-time, inference speed is critical. The semantic embeddings capture meaning well enough for safety classification.

**Q3: What is reward hacking and why is it dangerous in medical AI?**  
A: Reward hacking occurs when a model finds responses that score high on the reward model but are actually unsafe. In medical AI, this could mean a model learns to include safety keywords while still giving dangerous advice — potentially harming patients.

**Q4: How does the centroid-based approach work for scoring?**  
A: We compute the average embedding of all safe examples (safe centroid) and all unsafe examples (unsafe centroid). A new response is scored by comparing its embedding similarity to both centroids — responses closer to the safe centroid get higher scores.

**Q5: What safety threshold should the GAS system use for auto-delivery?**  
A: The GAS system uses score > 0.8 for auto-delivery, 0.5-0.8 for delivery with disclaimer, and < 0.5 for blocking and escalating to a physician. The threshold is conservative because false negatives (missing unsafe advice) are more dangerous than false positives (over-escalating).